# Lecture 2 Lab 1 — Numerical Two-Body vs J2 Propagation

This lab compares ideal **two-body** propagation to **J2-perturbed** propagation for a low Earth orbit.

## Learning Objectives
- Show the limitation of the two-body model.
- Implement two-body and J2-perturbed equations of motion.
- Compare orbital element evolution under both models.
- Demonstrate secular RAAN drift caused by Earth oblateness.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True

mu = 398600.4418
Re = 6378.1363
J2 = 1.08262668e-3

## 1) Utility Functions

In [ ]:
def rot3(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def rot1(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def keplerian_to_cartesian(a, e, i, raan, omega, nu, mu=mu):
    p = a * (1 - e**2)
    r_pf = (p / (1 + e * np.cos(nu))) * np.array([np.cos(nu), np.sin(nu), 0.0])
    v_pf = np.sqrt(mu / p) * np.array([-np.sin(nu), e + np.cos(nu), 0.0])
    Q_pX = rot3(raan) @ rot1(i) @ rot3(omega)
    r_eci = Q_pX @ r_pf
    v_eci = Q_pX @ v_pf
    return np.hstack((r_eci, v_eci))

def cartesian_to_keplerian(state, mu=mu):
    r = state[:3]
    v = state[3:]
    r_norm = np.linalg.norm(r)
    v_norm = np.linalg.norm(v)
    h = np.cross(r, v)
    h_norm = np.linalg.norm(h)
    k_hat = np.array([0.0, 0.0, 1.0])
    n = np.cross(k_hat, h)
    n_norm = np.linalg.norm(n)
    e_vec = (1/mu) * ((v_norm**2 - mu/r_norm)*r - np.dot(r, v)*v)
    e = np.linalg.norm(e_vec)
    energy = 0.5 * v_norm**2 - mu / r_norm
    a = -mu / (2*energy)
    i = np.arccos(np.clip(h[2]/h_norm, -1.0, 1.0))
    raan = 0.0 if n_norm < 1e-12 else np.arctan2(n[1], n[0])
    if n_norm < 1e-12 or e < 1e-12:
        omega = 0.0
    else:
        omega = np.arccos(np.clip(np.dot(n, e_vec)/(n_norm*e), -1.0, 1.0))
        if e_vec[2] < 0:
            omega = 2*np.pi - omega
    if e < 1e-12:
        nu = 0.0
    else:
        nu = np.arccos(np.clip(np.dot(e_vec, r)/(e*r_norm), -1.0, 1.0))
        if np.dot(r, v) < 0:
            nu = 2*np.pi - nu
    return a, e, i, raan, omega, nu

def two_body_rhs(t, y, mu=mu):
    r = y[:3]
    v = y[3:]
    r_norm = np.linalg.norm(r)
    a = -mu * r / r_norm**3
    return np.hstack((v, a))

def j2_rhs(t, y, mu=mu, Re=Re, J2=J2):
    r = y[:3]
    v = y[3:]
    x, y_, z = r
    r_norm = np.linalg.norm(r)
    a_tb = -mu * r / r_norm**3
    z2 = z*z
    r2 = r_norm*r_norm
    factor = 1.5 * J2 * mu * Re**2 / r_norm**5
    common = 5.0 * z2 / r2
    a_j2 = factor * np.array([x * (common - 1.0), y_ * (common - 1.0), z * (common - 3.0)])
    return np.hstack((v, a_tb + a_j2))

def propagate(rhs, y0, t_span, t_eval, rtol=1e-11, atol=1e-13):
    sol = solve_ivp(rhs, t_span=t_span, y0=y0, t_eval=t_eval, rtol=rtol, atol=atol, method='DOP853')
    if not sol.success:
        raise RuntimeError(sol.message)
    return sol

def unwrap_angle(angle_rad):
    return np.unwrap(angle_rad)

## 2) Initial Orbit and Propagation Setup

In [ ]:
alt_km = 700.0
a0 = Re + alt_km
e0 = 0.001
i0 = np.deg2rad(98.2)
raan0 = np.deg2rad(0.0)
omega0 = np.deg2rad(0.0)
nu0 = np.deg2rad(0.0)

y0 = keplerian_to_cartesian(a0, e0, i0, raan0, omega0, nu0, mu=mu)

days = 14
T_end = days * 24 * 3600.0
N = 5000

t_eval = np.linspace(0.0, T_end, N)
t_days = t_eval / 86400.0

In [ ]:
sol_tb = propagate(two_body_rhs, y0, (0.0, T_end), t_eval)
sol_j2 = propagate(j2_rhs, y0, (0.0, T_end), t_eval)

r_tb = sol_tb.y[:3].T
v_tb = sol_tb.y[3:].T
r_j2 = sol_j2.y[:3].T
v_j2 = sol_j2.y[3:].T

## 3) Convert Cartesian States Back to Orbital Elements

In [ ]:
def states_to_elements(r_arr, v_arr):
    a_list, e_list, i_list, raan_list, omega_list, nu_list = [], [], [], [], [], []
    for r, v in zip(r_arr, v_arr):
        a, e, inc, raan, omg, nu = cartesian_to_keplerian(np.hstack((r, v)), mu=mu)
        a_list.append(a)
        e_list.append(e)
        i_list.append(inc)
        raan_list.append(raan)
        omega_list.append(omg)
        nu_list.append(nu)
    return (np.array(a_list), np.array(e_list), np.array(i_list), np.array(raan_list), np.array(omega_list), np.array(nu_list))

(a_tb, e_tb, i_tb, raan_tb, omega_tb, nu_tb) = states_to_elements(r_tb, v_tb)
(a_j2, e_j2, i_j2, raan_j2, omega_j2, nu_j2) = states_to_elements(r_j2, v_j2)

raan_tb_u = unwrap_angle(raan_tb)
raan_j2_u = unwrap_angle(raan_j2)
omega_tb_u = unwrap_angle(omega_tb)
omega_j2_u = unwrap_angle(omega_j2)

## 4) Validation Checks

In [ ]:
energy_tb = 0.5*np.sum(v_tb**2, axis=1) - mu/np.linalg.norm(r_tb, axis=1)
energy_rel_err = (energy_tb - energy_tb[0]) / abs(energy_tb[0])
print(f"Two-body relative energy variation (max abs): {np.max(np.abs(energy_rel_err)):.3e}")

raan_drift_deg = np.rad2deg(raan_j2_u[-1] - raan_j2_u[0])
print(f"J2 RAAN net drift over {days} days: {raan_drift_deg:.3f} deg")
print("Expected sign for retrograde i=98.2 deg orbit: positive drift.")

## 5) Plots

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(r_tb[:,0], r_tb[:,1], r_tb[:,2], label='Two-body', lw=1)
ax.plot(r_j2[:,0], r_j2[:,1], r_j2[:,2], label='J2-perturbed', lw=1)
ax.set_xlabel('x [km]')
ax.set_ylabel('y [km]')
ax.set_zlabel('z [km]')
ax.set_title('3D Orbit Comparison (14 days)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12, 12), sharex=True)
axs[0,0].plot(t_days, a_tb, label='Two-body')
axs[0,0].plot(t_days, a_j2, label='J2', alpha=0.9)
axs[0,0].set_ylabel('a [km]'); axs[0,0].set_title('Semi-major axis'); axs[0,0].legend()

axs[0,1].plot(t_days, e_tb, label='Two-body')
axs[0,1].plot(t_days, e_j2, label='J2', alpha=0.9)
axs[0,1].set_ylabel('e [-]'); axs[0,1].set_title('Eccentricity')

axs[1,0].plot(t_days, np.rad2deg(i_tb), label='Two-body')
axs[1,0].plot(t_days, np.rad2deg(i_j2), label='J2', alpha=0.9)
axs[1,0].set_ylabel('i [deg]'); axs[1,0].set_title('Inclination')

axs[1,1].plot(t_days, np.rad2deg(raan_tb_u), label='Two-body')
axs[1,1].plot(t_days, np.rad2deg(raan_j2_u), label='J2', alpha=0.9)
axs[1,1].set_ylabel('RAAN [deg]'); axs[1,1].set_title('RAAN (unwrapped)')

axs[2,0].plot(t_days, np.rad2deg(omega_tb_u), label='Two-body')
axs[2,0].plot(t_days, np.rad2deg(omega_j2_u), label='J2', alpha=0.9)
axs[2,0].set_ylabel('ω [deg]'); axs[2,0].set_title('Argument of perigee (unwrapped)'); axs[2,0].set_xlabel('Time [days]')

dr = np.linalg.norm(r_j2 - r_tb, axis=1)
axs[2,1].plot(t_days, dr, color='k')
axs[2,1].set_ylabel('|Δr| [km]'); axs[2,1].set_title('Trajectory difference: |r_J2 - r_2body|'); axs[2,1].set_xlabel('Time [days]')

for ax in axs.flat:
    ax.grid(True)

plt.tight_layout()
plt.show()

## 6) Interpretation and Discussion

### Why J2 causes secular RAAN drift
Earth is oblate (equatorial bulge), not a perfect sphere. The **J2** term is the dominant correction to central gravity and creates a persistent nodal precession. For near-polar retrograde orbits (like 98.2°), this precession is typically **positive** (eastward), enabling sun-synchronous behavior.

### Why two-body propagation is insufficient for real missions
Two-body dynamics include only central gravity. Real trajectories are affected by non-spherical gravity, drag, third-body forces, solar radiation pressure, and maneuvers. Over days to weeks, omitting perturbations leads to significant node and along-track errors.

### Why osculating elements oscillate under perturbations
With perturbations present, the instantaneous Keplerian orbit tangent to the real trajectory changes continuously. Therefore osculating elements exhibit short-period oscillations plus long-term secular trends.

## 7) Industrial Interpretation

- **Sun-synchronous missions:** Designers tune altitude and inclination so J2-driven RAAN drift matches the required local solar time progression.
- **Ground-track repeatability:** J2 modifies nodal period and phase, so repeat-track planning must include perturbations.
- **Mission design and operations:** Accurate station-keeping budgets, imaging opportunities, and conjunction screening require perturbation-aware propagation.